In [ ]:
!pip install transformers==4.39.0 pandas tqdm torch einops huggingface_hub IndicTransToolkit

import transformers
# Fix for IndicTransToolkit's hardcoded import path
import transformers.tokenization_utils
if not hasattr(transformers.tokenization_utils, "PreTrainedTokenizerBase"):
    transformers.tokenization_utils.PreTrainedTokenizerBase = transformers.PreTrainedTokenizerBase

In [ ]:
from google.colab import drive
drive.mount('/content/drive')\
#



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# -------------------------------------------------------------
# CELL 1: Install Dependencies
# Run this cell first in Colab!
# -------------------------------------------------------------
# !pip install -q transformers==4.43.3 pandas tqdm torch einops huggingface_hub nltk sacremoses regex mock mosestokenizer sentencepiece
# !python3 -c "import nltk; nltk.download('punkt')"
# !git clone https://github.com/VarunGumma/IndicTransToolkit
# !cd IndicTransToolkit && pip install -e .

# -------------------------------------------------------------
# CELL 2: Mount Drive & Authenticate
# -------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

from huggingface_hub import login
HF_TOKEN = ""
login(token=HF_TOKEN)  # Your Read token!

# -------------------------------------------------------------
# CELL 3: The Translation Script
# -------------------------------------------------------------
import os
import gc
import pandas as pd
from tqdm import tqdm
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

# Your Drive path (assuming you made the shortcut to My Drive)
INPUT_DIR = "/content/drive/MyDrive/Bhaasha/split_data"
OUTPUT_DIR = "/content/drive/MyDrive/Bhaasha/split_data_translated"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Colab's Free T4 GPU is powerful. A batch size of 16-32 fits perfectly for a 1B model!
BATCH_SIZE = 16

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model_name = "ai4bharat/indictrans2-en-indic-1B"
print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, token=HF_TOKEN)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, trust_remote_code=True, torch_dtype="auto", token=HF_TOKEN).to(device)
ip = IndicProcessor(inference=True)
print("Model and Processor loaded successfully!")

def translate_batch_eng_to_mar(english_texts):
    if not english_texts:
        return []

    cleaned_texts = [text if (isinstance(text, str) and text.strip()) else " " for text in english_texts]

    try:
        # Preprocess the entire batch using IndicProcessor
        batch = ip.preprocess_batch(cleaned_texts, src_lang="eng_Latn", tgt_lang="mar_Deva")

        # Tokenize the processed batch
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)

        # Generate translations
        translated_tokens = model.generate(
            **inputs,
            max_length=512,
            num_beams=4
        )

        # Decode the generated tokens
        translated_texts = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)

        # Postprocess the text back to human readable Marathi
        translated_texts = ip.postprocess_batch(translated_texts, lang="mar_Deva")

        final_translations = []
        for orig, trans in zip(english_texts, translated_texts):
            if not isinstance(orig, str) or not orig.strip():
                final_translations.append("")
            else:
                final_translations.append(trans.strip())

        return final_translations

    except Exception as e:
        print(f"\nTranslation Error: {e}")
        return [""] * len(english_texts)

print("\nValidating paths before processing...")
if not os.path.exists(INPUT_DIR):
    print(f"\n❌ ERROR: Cannot find INPUT_DIR at '{INPUT_DIR}'")
    print("Did you upload the 'split_data' folder to Google Drive?")
    try:
        my_drive_contents = os.listdir('/content/drive/MyDrive')
        print("\n📂 Here is what is currently inside your Google Drive (/content/drive/MyDrive):")
        for item in my_drive_contents[:20]:  # Print first 20 items
            print(f"  - {item}")
        if 'Bhaasha' not in my_drive_contents:
            print("\n💡 HINT: The folder 'Bhaasha' is missing from your Google Drive.")
            print("Please create it and upload your local 'split_data' folder into it!")
    except Exception as e:
        print(f"Could not read Google Drive contents: {e}")
    raise FileNotFoundError(f"Input directory not found: {INPUT_DIR}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Input directory found: {INPUT_DIR}")

file_limits = {
    "train.csv": 200000,  # 2 Lakhs
    "test.csv": 50000,    # 50K
    "val.csv": 20000      # 20K
}

files_to_process = ["train.csv"]  # Focus only on train.csv first as requested

for filename in files_to_process:
    input_path = os.path.join(INPUT_DIR, filename)

    # Save with benchmark suffix as requested
    output_filename = filename.replace('.csv', '_benchmark.csv')
    output_path = os.path.join(OUTPUT_DIR, output_filename)

    if not os.path.exists(input_path):
        continue

    limit = file_limits.get(filename)

    print(f"\n--- Processing {filename} (Limit: {limit} rows) ---")
    chunksize = 100  # Process strictly 100 rows at a time
    first_chunk = not os.path.exists(output_path)

    # nrows limits the total rows read by pandas, while chunksize iterates through them in blocks
    chunk_iterator = pd.read_csv(input_path, chunksize=chunksize, nrows=limit)

    for chunk_idx, chunk in enumerate(chunk_iterator):
        print(f"Translating chunk {chunk_idx + 1} (Rows {chunk_idx*chunksize} - {(chunk_idx+1)*chunksize}) for {filename}...")

        translations = []
        english_data = chunk['english_legal_data'].fillna("").tolist()

        for i in tqdm(range(0, len(english_data), BATCH_SIZE), desc="Batches"):
            batch_texts = english_data[i:i + BATCH_SIZE]
            batch_translations = translate_batch_eng_to_mar(batch_texts)
            translations.extend(batch_translations)

        chunk['benchmark_translation'] = translations

        mode = 'a' if not first_chunk else 'w'
        header = first_chunk
        chunk.to_csv(output_path, mode=mode, header=header, index=False)
        first_chunk = False

        # --- CRITICAL MEMORY CLEANUP ---
        # Explicitly delete variables holding large structures
        del english_data
        del translations
        del chunk

        # Force Python to deep clean system RAM
        gc.collect()

        # Force PyTorch to wipe GPU VRAM
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(f"Finished writing {filename} to {output_path}")

print("\nProcessing complete! Files are saved in your Google Drive.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
Loading ai4bharat/indictrans2-en-indic-1B...


OutOfMemoryError: CUDA out of memory. Tried to allocate 128.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 115.81 MiB is free. Including non-PyTorch memory, this process has 14.45 GiB memory in use. Of the allocated memory 13.13 GiB is allocated by PyTorch, and 1.19 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)